In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder


df = pd.read_csv('/content/sample_data/CLEANecommerce_raw_dataset.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
df['month'] = df['order_date'].dt.to_period('M')


Ahora tenemos que normalizar los datos y realizar featuring engineering (crear nuevas variables) para poder alimentar un modelo de machine learning que nos ayude a predecir insights

Featuring Engineering Ideas:

•	revenue_per_unit = total_amount / quantity

•	is_discounted = 1 if discount_pct > 0 else 0

•	order_month = month number from order_date

•	order_dayofweek = day of week from order_date

•	is_returned = 1 if order_status == 'Returned' else 0 (your TARGET variable)


In [2]:
# --- Feature Engineering ---
df['is_discounted'] = (df['discount_pct'] > 0).astype(int)
df['order_month'] = df['order_date'].dt.month
df['order_dayofweek'] = df['order_date'].dt.dayofweek
df['revenue_per_unit'] = df['total_amount'] / df['quantity']
df['is_returned'] = (df['order_status'] == 'Returned').astype(int)
df['is_cancelled'] = (df['order_status'] == 'Cancelled').astype(int)


Normalizamos datos y creamos dos nuevas variables que son las trataremos de predecir:
- is_returned = si un producto es devuelto o no
- is_cancelled = si un producto es cancelado o no

Ahora para responder la otra pregunta adicional que creamos
QUESTION 8: ¿podemos predecir los meses con mas y menos ventas del año siguiente y podemos predecir el producto con mayor y menor ventas el año siguiente?

Tenemos que crear features adicionales y normalizar datos para realizar series de tiempo, específicamente las de tiempo, crearemos las features de:
- year
- quarter
- week of year
- is wekeend

- indice de tiempo numérico
- codificación cíclica del mes sen
- codificación cíclica del mes cos
- codificación cíclica del dia sen
- codificación cílcica del dia cos

In [3]:
# --- Features adicionales para forecasting ---

# Año y trimestre
df['year'] = df['order_date'].dt.year
df['quarter'] = df['order_date'].dt.quarter
df['week_of_year'] = df['order_date'].dt.isocalendar().week.astype('Int64') # Changed to nullable integer type
df['is_weekend'] = df['order_date'].dt.dayofweek.isin([5, 6]).astype(int)

# Índice de tiempo numérico (útil para capturar tendencia general)
df['days_since_start'] = (df['order_date'] - df['order_date'].min()).dt.days

# Codificación cíclica del mes (¡importante para ML!)
import numpy as np
df['month_sin'] = np.sin(2 * np.pi * df['order_month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['order_month'] / 12)

# Codificación cíclica del día de la semana
df['day_sin'] = np.sin(2 * np.pi * df['order_dayofweek'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['order_dayofweek'] / 7)

Antes de normalizar los datos vamos a descargar este nuevo archivo cvs para poder usar en power BI, es necesario descargar este nuevo con las nuevas feature sin las columnas normalizadas para poder graficar con sentido en power BI

In [5]:
# Agrega esto al final de tu 04_normalize.py, ANTES de aplicar StandardScaler:
df.to_csv('/content/sample_data/POWERBIecommerce_featured.csv', index=False)  # <- Este es tu archivo ideal para Power BI



Ahora normalizaremos algunas columnas

In [ ]:
# --- Encode categorical columns ---
le = LabelEncoder()
for col in ['category', 'gender', 'country', 'payment_method', 'shipping_method']:
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

# --- Normalize numeric columns ---
num_cols = ['age', 'quantity', 'unit_price', 'discount_pct',
            'total_amount', 'order_month', 'order_dayofweek']

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[num_cols] = scaler.fit_transform(df[num_cols])


Por ultimo, guardaremos el nuevo cvs

In [ ]:
df_scaled.to_csv('/content/sample_data/READYecommerce_raw_dataset.csv', index=False)
print('Model-ready dataset saved!')


Model-ready dataset saved!
